In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../..")

from src.classifical_features import build_monthly_features
from src.configs.settings import Settings
from src.visualization.visualization import (
    plot_best_predictions,
    plot_overall_metrics_comparison,
    plot_panel_metrics_distributions,
    plot_worst_predictions,
)

settings = Settings()

In [ ]:
data_dir = Path("../../data")

In [ ]:
features_df = pd.read_csv(data_dir / "processed" / "mirrors_clusters.csv")
mirrors_ts_df = pd.read_csv(data_dir / "processed" / "filtered_mirrors_ts.csv")

In [ ]:
features = build_monthly_features(mirrors_ts_df, settings)
features["panel_id"] = features[settings.columns.id].astype(str)

In [ ]:
from src.catboost_utilities.evaluate import evaluate_catboost
from src.catboost_utilities.train import train_catboost
from src.custom_types import CatBoostParameters
from src.model_selection import temporal_panel_split_by_size

regression_splits = temporal_panel_split_by_size(
    features,
    panel_column='article',
    time_column='date',
    test_size=3,
    val_size=None
)

In [ ]:
from dataclasses import dataclass

import pandas as pd

from src.custom_types import Splits


@dataclass
class PanelSplitStats:
    """Статистики панели по сплитам."""
    
    panel_id: str
    train_mean: float
    test_mean: float
    train_std: float
    test_std: float
    mean_diff: float
    std_ratio: float


def compute_panel_split_stats(
    splits: Splits[pd.DataFrame],
    panel_column: str,
    target_column: str,
) -> list[PanelSplitStats]:
    """Вычисляет статистики по панелям для train/test сплитов."""
    train_stats = splits.train.groupby(panel_column)[target_column].agg(['mean', 'std'])
    test_stats = splits.test.groupby(panel_column)[target_column].agg(['mean', 'std'])
    
    result = []
    for panel_id in train_stats.index:
        if panel_id not in test_stats.index:
            continue
        
        train_mean, train_std = train_stats.loc[panel_id]
        test_mean, test_std = test_stats.loc[panel_id]
        
        result.append(PanelSplitStats(
            panel_id=panel_id,
            train_mean=train_mean,
            test_mean=test_mean,
            train_std=train_std,
            test_std=test_std,
            mean_diff=abs(test_mean - train_mean),
            std_ratio=test_std / train_std if train_std > 0 else float('inf'),
        ))
    
    return result


def find_ood_panels(
    splits: Splits[pd.DataFrame],
    panel_column: str,
    target_column: str,
    mean_diff_threshold: float | None = None,
    std_ratio_threshold: float | None = None,
    z_score_threshold: float | None = 2.0,
) -> list[PanelSplitStats]:
    """Находит панели с out-of-distribution test данными."""
    stats = compute_panel_split_stats(splits, panel_column, target_column)
    
    ood_panels = []
    for s in stats:
        is_ood = False
        
        if mean_diff_threshold is not None and s.mean_diff > mean_diff_threshold:
            is_ood = True
        
        if std_ratio_threshold is not None:
            if s.std_ratio > std_ratio_threshold or s.std_ratio < 1 / std_ratio_threshold:
                is_ood = True
        
        if z_score_threshold is not None and s.train_std > 0:
            z_score = s.mean_diff / s.train_std
            if z_score > z_score_threshold:
                is_ood = True
        
        if is_ood:
            ood_panels.append(s)
    
    return sorted(ood_panels, key=lambda x: x.mean_diff, reverse=True)


def filter_ood_panels(
    splits: Splits[pd.DataFrame],
    panel_column: str,
    target_column: str,
    mean_diff_threshold: float | None = None,
    std_ratio_threshold: float | None = None,
    z_score_threshold: float | None = 2.0,
    print_stats: bool = False,
) -> tuple[Splits[pd.DataFrame], list[PanelSplitStats]]:
    """Фильтрует панели с OOD test данными и возвращает отфильтрованные сплиты."""
    ood_panels = find_ood_panels(
        splits, panel_column, target_column,
        mean_diff_threshold, std_ratio_threshold, z_score_threshold
    )
    
    ood_ids = {s.panel_id for s in ood_panels}
    
    def _filter_df(df: pd.DataFrame) -> pd.DataFrame:
        return df[~df[panel_column].isin(ood_ids)].reset_index(drop=True)
    
    filtered_splits = splits.apply(_filter_df)
    
    if print_stats:
        total_panels = splits.train[panel_column].nunique()
        filtered_count = len(ood_panels)
        remaining = total_panels - filtered_count
        print(
            f"OOD фильтрация: {filtered_count}/{total_panels} панелей удалено "
            f"({filtered_count / total_panels * 100:.1f}%), осталось {remaining}"
        )
    
    return filtered_splits, ood_panels

In [ ]:
filtered_splits, ood_panels = filter_ood_panels(
    regression_splits,
    panel_column='article',
    target_column='sales',
    z_score_threshold=2.0,
    print_stats=True,
)

pd.DataFrame([vars(p) for p in ood_panels])

In [ ]:
from src.data_processing import scale_panel_splits

scale = not settings.downstream.round_predictions
if scale:
    splits = scale_panel_splits(
        (data for _, data in regression_splits.splits),
        panel_column=settings.columns.id,
        target_columns=settings.columns.main_target,
        apply_log=settings.preprocessing.apply_log,
    )
else:
    splits = regression_splits

In [ ]:
catboost_params = CatBoostParameters(
    iterations=500,
    learning_rate=0.05,
    depth=3,
    l2_leaf_reg=10,
    subsample=0.8,
    rsm=0.8,
    random_seed=settings.random_state,
    verbose=50,
    loss_function="RMSE",
)

catboost_model = train_catboost(
    train_df=splits.train,
    val_df=splits.val,
    params=catboost_params,
    settings=settings,
)

results = evaluate_catboost(
    model=catboost_model,
    splits=splits,
    settings=settings,
    scalers=None,
)

In [ ]:
overall_df = results.get_overall_metrics_df()
panel_df = results.get_panel_metrics_df()


plot_overall_metrics_comparison(results)

plot_panel_metrics_distributions(
    results=results,
    metrics_to_plot=["mape", "rmse", "mae", "r2"],
)

In [ ]:
plot_best_predictions(
    results=results,
    n_best=5,
    metric="rmse",
    split_name="test",
)

In [ ]:
plot_worst_predictions(
    results=results,
    n_worst=5,
    metric="rmse",
    split_name="test",
)

In [ ]:
feature_importance = catboost_model.get_feature_importance()
feature_names = regression_splits.train.drop(
    columns=[settings.columns.main_target, settings.columns.id, settings.columns.date]
).columns

importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": feature_importance,
    }
).sort_values("importance", ascending=False)

importance_df